# 05 — Advanced Analytics

Value at Risk (VaR), Cohort Analysis, and Fund Recommender System.

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.dates as mdatesimport seaborn as snsfrom scipy import statsimport warningsimport oswarnings.filterwarnings('ignore')os.chdir('/Users/maneeshreddy/Desktop/mutual_fund_analysis')plt.style.use('seaborn-v0_8-whitegrid')DATA_DIR = 'data_1/Bluestock_MF_Datasets'REPORTS_DIR = 'reports/figures'RF = 0.065TRADING_DAYS = 252print('Libraries loaded.')

In [ ]:
nav_df = pd.read_csv(DATA_DIR + '/02_nav_history.csv', parse_dates=['date'])perf_df = pd.read_csv(DATA_DIR + '/07_scheme_performance.csv')txn_df = pd.read_csv(DATA_DIR + '/08_investor_transactions.csv', parse_dates=['transaction_date'])fm_df = pd.read_csv(DATA_DIR + '/01_fund_master.csv')nav_pivot = nav_df.pivot(index='date', columns='amfi_code', values='nav').sort_index()daily_returns = nav_pivot.pct_change().dropna()code_to_name = dict(zip(fm_df['amfi_code'], fm_df['scheme_name']))nav_pivot.columns = [code_to_name.get(c, str(c)) for c in nav_pivot.columns]daily_returns.columns = nav_pivot.columnsprint('Data loaded:', len(nav_pivot.columns), 'funds,', len(nav_pivot), 'days')

## Part 1: Value at Risk (VaR)

Compute Parametric VaR, Historical VaR, and Conditional VaR (CVaR) at 95% and 99% confidence.

In [ ]:
def compute_var(returns, confidence_levels=[0.95, 0.99]):    results = {}    mu = returns.mean()    sigma = returns.std()        for cl in confidence_levels:        alpha = 1 - cl        z = stats.norm.ppf(alpha)        var_parametric = -(mu + z * sigma)        var_historical = -np.percentile(returns.dropna(), alpha * 100)        cvar = -returns[returns <= -var_historical].mean()        results['VaR_param_' + str(int(cl*100))] = var_parametric        results['VaR_hist_' + str(int(cl*100))] = var_historical        results['CVaR_' + str(int(cl*100))] = cvar    return resultsvar_results = pd.DataFrame()for col in daily_returns.columns:    var = compute_var(daily_returns[col])    for k, v in var.items():        var_results.loc[col, k] = vvar_results['VaR_param_95_ann'] = var_results['VaR_param_95'] * np.sqrt(TRADING_DAYS)var_results['VaR_param_99_ann'] = var_results['VaR_param_99'] * np.sqrt(TRADING_DAYS)var_results = var_results.sort_values('VaR_param_95')print('Value at Risk (Daily %) - Top 10 safest:')var_results.head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))for ax, col, title in zip(axes,    ['VaR_param_95', 'VaR_hist_95', 'CVaR_95'],    ['Parametric VaR 95%', 'Historical VaR 95%', 'CVaR 95%']):    sorted_data = var_results[col].sort_values(ascending=True).head(20)    sorted_data.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')    ax.set_title(title, fontsize=12, fontweight='bold')    ax.set_xlabel('Daily Loss (%)')    ax.tick_params(labelsize=7)plt.suptitle('Value at Risk Comparison - Top 20 Safest Funds', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig(REPORTS_DIR + '/20_var_comparison.png', dpi=150, bbox_inches='tight')plt.show()

In [ ]:
top5 = var_results.head(5).index.tolist()fig, axes = plt.subplots(1, 5, figsize=(25, 5))for ax, fund in zip(axes, top5):    data = daily_returns[fund].dropna()    ax.hist(data, bins=80, alpha=0.7, color='steelblue', density=True, edgecolor='white', linewidth=0.3)    var95 = var_results.loc[fund, 'VaR_hist_95']    var99 = var_results.loc[fund, 'VaR_hist_99']    ax.axvline(-var95, color='orange', linewidth=1.5, linestyle='--', label='95% VaR')    ax.axvline(-var99, color='red', linewidth=1.5, linestyle='--', label='99% VaR')    ax.set_title(fund[:30] + '...', fontsize=8)    ax.legend(fontsize=7)    ax.tick_params(labelsize=6)plt.suptitle('Return Distributions with VaR Thresholds', fontsize=13, fontweight='bold')plt.tight_layout()plt.savefig(REPORTS_DIR + '/21_var_distributions.png', dpi=150, bbox_inches='tight')plt.show()

## Part 2: Cohort Analysis

Analyze investor behavior by cohort (age group, city tier, gender).

In [ ]:
print('Total transactions:', len(txn_df))print('Unique investors:', txn_df['investor_id'].nunique())print()print('Age group distribution:')print(txn_df['age_group'].value_counts())print()print('City tier distribution:')print(txn_df['city_tier'].value_counts())print()print('Gender distribution:')print(txn_df['gender'].value_counts())print()print('Transaction type distribution:')print(txn_df['transaction_type'].value_counts())

In [ ]:
# Age group cohort using dict-based aggage_cohort = txn_df.groupby('age_group').agg(    txn_count=('amount_inr', 'count'),    unique_investors=('investor_id', 'nunique'),    total_amount=('amount_inr', 'sum'),    avg_amount=('amount_inr', 'mean'),    median_amount=('amount_inr', 'median'),    avg_income=('annual_income_lakh', 'mean')).round(2)age_cohort['pct_of_total'] = (age_cohort['total_amount'] / age_cohort['total_amount'].sum() * 100).round(1)age_cohort = age_cohort.reindex(['18-25', '26-35', '36-45', '46-55', '56+'])print('Age Group Cohort Analysis:')age_cohort

In [ ]:
city_cohort = txn_df.groupby('city_tier').agg(    txn_count=('amount_inr', 'count'),    unique_investors=('investor_id', 'nunique'),    total_amount=('amount_inr', 'sum'),    avg_amount=('amount_inr', 'mean')).round(2)city_cohort['pct_of_total'] = (city_cohort['total_amount'] / city_cohort['total_amount'].sum() * 100).round(1)print('City Tier Cohort Analysis:')city_cohort

In [ ]:
cross = txn_df.groupby(['age_group', 'gender'])['amount_inr'].sum().unstack(fill_value=0)cross = cross.reindex(['18-25', '26-35', '36-45', '46-55', '56+'])fig, axes = plt.subplots(1, 3, figsize=(22, 6))sns.heatmap(cross / 1e6, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[0], cbar_kws={'label': 'Cr INR'})axes[0].set_title('Total Investment: Age x Gender', fontweight='bold')axes[0].set_ylabel('Age Group')cross2 = txn_df.groupby(['age_group', 'city_tier'])['amount_inr'].mean().unstack(fill_value=0)cross2 = cross2.reindex(['18-25', '26-35', '36-45', '46-55', '56+'])sns.heatmap(cross2, annot=True, fmt='.0f', cmap='YlGnBu', ax=axes[1], cbar_kws={'label': 'INR'})axes[1].set_title('Avg Transaction: Age x City Tier', fontweight='bold')axes[1].set_ylabel('')cross3 = txn_df.groupby(['transaction_type', 'age_group'])['amount_inr'].count().unstack(fill_value=0)cross3 = cross3.reindex(columns=['18-25', '26-35', '36-45', '46-55', '56+'])sns.heatmap(cross3, annot=True, fmt='d', cmap='Purples', ax=axes[2], cbar_kws={'label': 'Count'})axes[2].set_title('Transaction Count: Type x Age', fontweight='bold')axes[2].set_ylabel('')plt.suptitle('Cohort Analysis Heatmaps', fontsize=14, fontweight='bold', y=1.02)plt.tight_layout()plt.savefig(REPORTS_DIR + '/22_cohort_heatmaps.png', dpi=150, bbox_inches='tight')plt.show()

In [ ]:
sip_txns = txn_df[txn_df['transaction_type'] == 'SIP'].copy()sip_txns = sip_txns.sort_values(['investor_id', 'transaction_date'])sip_counts = sip_txns.groupby('investor_id').agg(    sip_count=('amount_inr', 'count'),    first_sip=('transaction_date', 'min'),    last_sip=('transaction_date', 'max'),    total_sip_amount=('amount_inr', 'sum'))sip_counts['sip_tenure_days'] = (sip_counts['last_sip'] - sip_counts['first_sip']).dt.daysfig, axes = plt.subplots(1, 3, figsize=(20, 6))axes[0].hist(sip_counts['sip_count'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)axes[0].set_title('SIP Frequency Distribution', fontweight='bold')axes[0].set_xlabel('Number of SIP Transactions')axes[0].set_ylabel('Number of Investors')axes[1].hist(sip_counts['sip_tenure_days'].dropna(), bins=30, color='coral', edgecolor='white', alpha=0.8)axes[1].set_title('SIP Tenure Distribution', fontweight='bold')axes[1].set_xlabel('Tenure (Days)')axes[1].set_ylabel('Number of Investors')axes[2].scatter(sip_counts['sip_count'], sip_counts['total_sip_amount'], alpha=0.3, s=10, color='green')axes[2].set_title('SIP Amount vs Frequency', fontweight='bold')axes[2].set_xlabel('Number of SIP Transactions')axes[2].set_ylabel('Total SIP Amount (INR)')plt.suptitle('Investor Retention Analysis', fontsize=14, fontweight='bold', y=1.02)plt.tight_layout()plt.savefig(REPORTS_DIR + '/23_retention_analysis.png', dpi=150, bbox_inches='tight')plt.show()print('SIP investors:', len(sip_counts))print('Avg SIP frequency:', round(sip_counts['sip_count'].mean(), 1))print('Avg SIP tenure (days):', round(sip_counts['sip_tenure_days'].mean(), 0))

## Part 3: Fund Recommender System

Content-based filtering using fund characteristics.

In [ ]:
features = perf_df[['amfi_code', 'scheme_name', 'category', 'return_1yr_pct',    'return_3yr_pct', 'sharpe_ratio', 'sortino_ratio', 'beta',    'expense_ratio_pct', 'aum_crore', 'morningstar_rating',    'max_drawdown_pct', 'risk_grade']].copy()risk_map = {'Low': 1, 'Moderate': 2, 'Moderately High': 3, 'High': 4, 'Very High': 5}features['risk_score'] = features['risk_grade'].map(risk_map)from sklearn.preprocessing import MinMaxScalerscaler = MinMaxScaler()numeric_cols = ['return_1yr_pct', 'return_3yr_pct', 'sharpe_ratio', 'sortino_ratio',    'beta', 'expense_ratio_pct', 'aum_crore', 'morningstar_rating',    'max_drawdown_pct', 'risk_score']features[numeric_cols] = features[numeric_cols].fillna(features[numeric_cols].median())features_scaled = pd.DataFrame(    scaler.fit_transform(features[numeric_cols]),    columns=numeric_cols,    index=features.index)print('Feature matrix shape:', features_scaled.shape)features.head()

In [ ]:
from sklearn.metrics.pairwise import cosine_similaritydef recommend_funds(fund_name, n=5):    mask = features['scheme_name'].str.contains(fund_name, case=False, na=False)    if not mask.any():        print('Fund not found.')        return None    idx = mask.idxmax()    fund_name_full = features.loc[idx, 'scheme_name']    sim_matrix = cosine_similarity(features_scaled)    sim_scores = list(enumerate(sim_matrix[idx]))    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)    top_indices = [i[0] for i in sim_scores[1:n+1]]    result = features.iloc[top_indices][['scheme_name', 'category', 'return_3yr_pct',        'sharpe_ratio', 'expense_ratio_pct', 'risk_grade']].copy()    result['similarity_score'] = [sim_scores[i][1] for i in range(1, n+1)]    print('Recommendations for:', fund_name_full)    return resultfor test in ['SBI Bluechip', 'HDFC Mid-Cap', 'Axis Small Cap']:    print()    recs = recommend_funds(test, n=3)    if recs is not None:        print(recs.to_string(index=False))

In [ ]:
def recommend_by_profile(risk_tolerance, investment_horizon, top_n=5):    risk_filters = {        'conservative': ['Low', 'Moderate'],        'moderate': ['Moderate', 'Moderately High'],        'aggressive': ['High', 'Very High']    }    filtered = features[features['risk_grade'].isin(risk_filters[risk_tolerance])].copy()        if investment_horizon == 'short':        filtered['score'] = (            0.3 * filtered['sharpe_ratio'].rank(pct=True) +            0.3 * (1 - filtered['expense_ratio_pct'].rank(pct=True)) +            0.2 * filtered['morningstar_rating'].rank(pct=True) +            0.2 * filtered['max_drawdown_pct'].rank(pct=True)        )    elif investment_horizon == 'medium':        filtered['score'] = (            0.3 * filtered['return_3yr_pct'].rank(pct=True) +            0.25 * filtered['sharpe_ratio'].rank(pct=True) +            0.2 * (1 - filtered['expense_ratio_pct'].rank(pct=True)) +            0.15 * filtered['morningstar_rating'].rank(pct=True) +            0.1 * filtered['max_drawdown_pct'].rank(pct=True)        )    else:        filtered['score'] = (            0.35 * filtered['return_3yr_pct'].rank(pct=True) +            0.25 * filtered['sharpe_ratio'].rank(pct=True) +            0.15 * (1 - filtered['expense_ratio_pct'].rank(pct=True)) +            0.15 * filtered['morningstar_rating'].rank(pct=True) +            0.1 * filtered['max_drawdown_pct'].rank(pct=True)        )        result = filtered.nlargest(top_n, 'score')[['scheme_name', 'category', 'risk_grade',        'return_3yr_pct', 'sharpe_ratio', 'expense_ratio_pct', 'score']]    print('Profile:', risk_tolerance, '|', investment_horizon, 'term')    return resultfor risk in ['conservative', 'moderate', 'aggressive']:    for horizon in ['short', 'medium', 'long']:        print()        recs = recommend_by_profile(risk, horizon, top_n=3)        print(recs.to_string(index=False))

In [ ]:
var_results.to_csv('var_results.csv')print('var_results.csv saved.')print('Advanced Analytics complete.')